# Phase 2: Long-Tailed Learning Techniques
# Includes: LDAM Loss, DRW, Class-Balanced Sampling, Two-Stage cRT Training

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from config import *
from dataloader import load_and_split_data, create_dataloaders, get_class_counts, get_pos_weights, CXRDataset, get_train_transforms, get_val_transforms, IMAGE_DIR
from models.resnet import ResNet50Classifier
from losses import get_loss_function
from samplers import get_sampler, ClassBalancedSampler
from trainer import train_model, train_two_stage, validate, load_checkpoint
from utils import set_seed, create_submission, validate_submission
from torch.utils.data import DataLoader

print(f"Device: {DEVICE}")
set_seed(RANDOM_SEED)

In [ ]:
# Load data
train_df, val_df = load_and_split_data()
class_counts = get_class_counts(train_df)
labels_array = train_df[CLASS_NAMES].values

print(f"\nClass imbalance ratio: {class_counts.max() / class_counts.min():.1f}:1")
print(f"Max class: {CLASS_NAMES[class_counts.argmax()]} ({int(class_counts.max())})")
print(f"Min class: {CLASS_NAMES[class_counts.argmin()]} ({int(class_counts.min())})")

## Option A: LDAM + DRW Loss (Single-Stage with Deferred Re-Weighting)

In [ ]:
# Create dataloaders (standard sampling)
train_loader, val_loader = create_dataloaders(train_df, val_df)

# Initialize model
model_ldam = ResNet50Classifier(num_classes=NUM_CLASSES, pretrained=True, dropout=0.5)
model_ldam = model_ldam.to(DEVICE)

# LDAM + DRW Loss
# DRW starts applying class-balanced weights after drw_epoch
DRW_EPOCH = 15  # First 15 epochs: uniform weights, after: class-balanced

criterion_ldam_drw = get_loss_function(
    "ldam_drw", 
    class_counts=class_counts, 
    drw_epoch=DRW_EPOCH
)

optimizer = optim.AdamW(model_ldam.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print(f"Loss: LDAM + DRW (DRW starts at epoch {DRW_EPOCH})")
print(f"Total epochs: {NUM_EPOCHS}")

In [ ]:
# Train with LDAM + DRW
history_ldam, best_mAP_ldam = train_model(
    model=model_ldam,
    train_loader=val_loader,
    val_loader=val_loader,
    criterion=criterion_ldam_drw,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    early_stopping_patience=10,
    model_name="resnet50_ldam_drw"
)

print(f"\nLDAM+DRW Best mAP: {best_mAP_ldam:.4f}")

## Option B: Two-Stage Training (cRT - Classifier Re-Training)

In [ ]:
# Create class-balanced sampler for Stage 2
sampler_balanced = ClassBalancedSampler(labels_array, class_counts, num_samples=len(train_df))

# Create datasets
train_dataset = CXRDataset(train_df, IMAGE_DIR, transform=get_train_transforms(), is_test=False)
val_dataset = CXRDataset(val_df, IMAGE_DIR, transform=get_val_transforms(), is_test=False)

# Standard loader for Stage 1
train_loader_standard = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)

# Balanced loader for Stage 2
train_loader_balanced = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler_balanced,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f"Standard loader batches: {len(train_loader_standard)}")
print(f"Balanced loader batches: {len(train_loader_balanced)}")

In [ ]:
# Initialize model for two-stage training
model_crt = ResNet50Classifier(num_classes=NUM_CLASSES, pretrained=True, dropout=0.5)
model_crt = model_crt.to(DEVICE)

# Stage 1: Standard Focal Loss
criterion_stage1 = get_loss_function("focal")

# Stage 2: Class-Balanced Focal Loss
criterion_stage2 = get_loss_function("cb_focal", class_counts=class_counts)

optimizer = optim.AdamW(model_crt.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)

print("Two-Stage cRT Training:")
print("  Stage 1: Focal Loss + Standard Sampling (20 epochs)")
print("  Stage 2: CB-Focal Loss + Balanced Sampling (10 epochs, frozen backbone)")

In [ ]:
# Run two-stage training
history_crt, best_mAP_crt = train_two_stage(
    model=model_crt,
    train_loader=train_loader_standard,
    val_loader=val_loader,
    train_loader_balanced=train_loader_balanced,
    criterion_stage1=criterion_stage1,
    criterion_stage2=criterion_stage2,
    optimizer=optimizer,
    scheduler=scheduler,
    stage1_epochs=20,
    stage2_epochs=10,
    early_stopping_patience=7,
    model_name="resnet50_crt"
)

print(f"\ncRT Best mAP: {best_mAP_crt:.4f}")

## Option C: Asymmetric Loss + Square Root Sampling

In [ ]:
# Square-root sampling is a middle ground between natural and balanced
from samplers import SquareRootSampler

sampler_sqrt = SquareRootSampler(labels_array, class_counts, num_samples=len(train_df))

train_loader_sqrt = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler_sqrt,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)

# Asymmetric Loss - specifically designed for multi-label long-tailed
model_asym = ResNet50Classifier(num_classes=NUM_CLASSES, pretrained=True, dropout=0.5)
model_asym = model_asym.to(DEVICE)

criterion_asym = get_loss_function("asymmetric")
optimizer = optim.AdamW(model_asym.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print("Asymmetric Loss + Square Root Sampling")
print("  gamma_neg=4, gamma_pos=1, clip=0.05")

In [ ]:
# Train with Asymmetric Loss
history_asym, best_mAP_asym = train_model(
    model=model_asym,
    train_loader=train_loader_sqrt,
    val_loader=val_loader,
    criterion=criterion_asym,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    early_stopping_patience=10,
    model_name="resnet50_asymmetric"
)

print(f"\nAsymmetric Loss Best mAP: {best_mAP_asym:.4f}")

## Compare Results and Select Best Model

In [ ]:
# Compare all methods
results = {
    'LDAM + DRW': best_mAP_ldam,
    'Two-Stage cRT': best_mAP_crt,
    'Asymmetric + SqrtSampling': best_mAP_asym
}

print("\n" + "="*50)
print("PHASE 2 RESULTS COMPARISON")
print("="*50)
for method, mAP in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {method}: {mAP:.4f}")

best_method = max(results, key=results.get)
print(f"\nBest Method: {best_method} (mAP: {results[best_method]:.4f})")

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# LDAM+DRW
axes[0].plot(history_ldam['mAP'], label='LDAM+DRW')
axes[0].set_title(f"LDAM+DRW (Best: {best_mAP_ldam:.4f})")
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('mAP')

# cRT
axes[1].plot(history_crt['mAP'], label='cRT')
axes[1].axvline(x=len([s for s in history_crt['stage'] if s==1]), color='r', linestyle='--', label='Stage 2 Start')
axes[1].set_title(f"Two-Stage cRT (Best: {best_mAP_crt:.4f})")
axes[1].set_xlabel('Epoch')
axes[1].legend()

# Asymmetric
axes[2].plot(history_asym['mAP'], label='Asymmetric')
axes[2].set_title(f"Asymmetric Loss (Best: {best_mAP_asym:.4f})")
axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

## Evaluate Best Model Per-Class Performance

In [ ]:
# Load the best model based on results
# Update this based on which method performed best
best_model_name = "resnet50_crt"  # Change based on comparison results

best_model = ResNet50Classifier(num_classes=NUM_CLASSES, pretrained=False)
load_checkpoint(best_model, None, os.path.join(CHECKPOINT_DIR, f"{best_model_name}_best.pth"))
best_model = best_model.to(DEVICE)

# Evaluate
val_metrics = validate(best_model, val_loader, criterion_stage2)

# Per-class AP analysis: Compare head vs tail classes
print("\nPer-Class AP Analysis:")
print("-" * 50)

# Sort classes by frequency
sorted_classes = sorted(zip(CLASS_NAMES, class_counts, val_metrics['per_class_ap']), 
                        key=lambda x: -x[1])

# Head classes (top 10)
print("\nHEAD CLASSES (Top 10 by frequency):")
for name, count, ap in sorted_classes[:10]:
    print(f"  {name}: AP={ap:.4f} (n={int(count)})")

# Tail classes (bottom 10)
print("\nTAIL CLASSES (Bottom 10 by frequency):")
for name, count, ap in sorted_classes[-10:]:
    print(f"  {name}: AP={ap:.4f} (n={int(count)})")

# Compare head vs tail average AP
head_ap = np.mean([x[2] for x in sorted_classes[:10]])
tail_ap = np.mean([x[2] for x in sorted_classes[-10:]])
print(f"\nHead AP avg: {head_ap:.4f}")
print(f"Tail AP avg: {tail_ap:.4f}")
print(f"Improvement needed in tail: {head_ap - tail_ap:.4f}")

## Generate Test Predictions

In [ ]:
from dataloader import create_test_dataloader
from trainer import predict

test_loader, test_df = create_test_dataloader()
print(f"Test samples: {len(test_df)}")

# Generate predictions with best model
image_ids, predictions = predict(best_model, test_loader)

# Create submission
submission_df = create_submission(image_ids, predictions, f"submission_phase2_{best_model_name}.csv")
validate_submission(os.path.join(OUTPUT_DIR, f"submission_phase2_{best_model_name}.csv"))

In [ ]:
print("\n" + "="*50)
print("PHASE 2 COMPLETE!")
print("="*50)
print(f"Best Method: {best_method}")
print(f"Best mAP: {results[best_method]:.4f}")
print(f"Submission: outputs/submission_phase2_{best_model_name}.csv")